<a href="https://colab.research.google.com/github/hajonghyun/installPytorch_study/blob/main/7_1_mininet_test_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn

In [2]:
x = torch.randn(100,3)
layer = nn.Linear(3,5)

# print하기 전에 결과 예측해보기
print(layer(x).shape)
print(layer.weight.shape)
print(layer.bias.shape)


torch.Size([100, 5])
torch.Size([5, 3])
torch.Size([5])


# 딥러닝 핵심: ReLU (Rectified Linear Unit)

**ReLU**는 딥러닝 역사상 가장 중요한 발견 중 하나로, 딥러닝 모델을 깊게(Deep) 쌓을 수 있게 만든 핵심 활성화 함수입니다.

---

## 1. 💡 직관적 이해: "깐깐한 문지기"

ReLU는 들어오는 신호를 선별하는 **전기 스위치(Switch)** 또는 **문지기**와 같습니다.

* **양수 ($+$)**: "중요한 신호(Feature)다!" $\rightarrow$ **그대로 통과 (Pass)**
* **음수 ($-$)**: "쓸모없는 노이즈다." $\rightarrow$ **0으로 차단 (Block)**

> **왜 비선형(Non-linear)인가?**
> 세상의 데이터(이미지, 소리)는 직선 하나로 나눌 수 없습니다. 0에서 꺾이는(Rectified) 구조가 있어야 신경망이 복잡한 경계선을 그릴 수 있습니다.

---

## 2. 📐 수식과 그래프

$$f(x) = \max(0, x)$$

* $x > 0$: 기울기(Gradient) = **1**
* $x < 0$: 기울기(Gradient) = **0**

---

## 3. ❓ 핵심 Q&A (헷갈리기 쉬운 포인트)

이번 학습 과정에서 다루었던 중요한 오개념들을 정리합니다.

### Q1. "음수는 왜 불필요한 정보로 취급하나요?"
수학적으로 음수가 나쁜 것은 아니지만, **생물학적/구조적 효율성** 때문입니다.
1.  **생물학적 모방:** 뇌세포(뉴런)는 자극이 없으면 가만히 있지, 음수 신호를 쏘지 않습니다. (Firing or Not)
2.  **특징 추출(Feature Detection):** CNN은 "특징이 있냐/없냐"를 따집니다. 특징의 반대 패턴(음수)은 "특징 없음(0)"으로 처리하는 것이 모델을 단순화시킵니다.
3.  **희소성(Sparsity):** 적당히 많은 뉴런이 0이 되어야(꺼져야), 진짜 중요한 뉴런의 신호가 돋보입니다.

### Q2. "Leaky ReLU는 음수를 양수로 만드는 건가요?"
**아닙니다!** 부호는 그대로 두고 크기만 줄이는 것입니다.
* **오해:** $-100 \times -0.01 = +1$ (양수로 변환? ❌)
* **진실:** $-100 \times +0.01 = -1$ (음수 유지, 크기 축소 ⭕)
    * "아니라는 건 알겠는데(음수), 너무 강하게 부정하지는 마(값 축소)."라는 의미입니다.

---

## 4. 🏆 이론적 핵심: "Why ReLU?"

왜 Sigmoid를 버리고 ReLU를 쓸까요? 바로 **Vanishing Gradient (기울기 소실)** 문제 해결 때문입니다.

| 비교 | Sigmoid | ReLU |
| :--- | :--- | :--- |
| **최대 기울기** | **0.25** (너무 작음) | **1** (양수 구간) |
| **문제점** | 층이 깊어지면 $0.25 \times 0.25 \dots$ 반복하여 기울기가 0이 됨 (학습 마비). | $1 \times 1 \dots = 1$. 층이 아무리 깊어도 **기울기가 그대로 전달됨.** |
| **결과** | 얕은 모델만 가능 | **Deep Neural Network 가능** |

---

## 5. 💻 PyTorch 구현 (Code)

실무에서 사용하는 두 가지 방식입니다.

### 1) `nn.ReLU` (Class 방식)
* 주로 `nn.Sequential` 모델을 쌓을 때 레고 블록처럼 사용합니다.
* `inplace=True`: 메모리 절약을 위해 원본 데이터를 덮어씁니다.

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),  # 레이어로서 존재
    nn.MaxPool2d(2)
)
```

### 2) `F.relu` (Functional 방식)
* `forward` 함수 내에서 수식을 직접 짤 때 사용합니다.

```python
import torch.nn.functional as F

class MyModel(nn.Module):
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)  # 함수처럼 호출
        return x
```

---

## 6. ⚠️ 심화: Dying ReLU & Leaky ReLU

* **Dying ReLU:** 학습 중 뉴런에 계속 음수만 들어오면, 기울기가 0이 되어 영원히 깨어나지 못하는 "죽은 뉴런"이 발생할 수 있습니다.
* **Leaky ReLU:** 이를 방지하기 위해 음수 구간에 아주 미세한 기울기($0.01x$)를 주어 "살려는 드리는" 변형 함수입니다.

In [3]:
x=torch.randn(2,5)
layer=nn.ReLU()
print(x)
print(layer(x))

tensor([[ 0.0241,  1.4983, -0.8948, -0.3352,  0.5711],
        [ 0.5750,  1.1977,  0.2510,  0.0304, -0.5164]])
tensor([[0.0241, 1.4983, 0.0000, 0.0000, 0.5711],
        [0.5750, 1.1977, 0.2510, 0.0304, 0.0000]])


# 📚 Batch Normalization (BN) 완전 정복: A to Z

> **핵심 요약:** > "앞사람(이전 Layer)이 데이터를 아무렇게나 던져도, 내가 받아서 **깔끔하게 정렬(Normalize)**한 뒤 **가장 학습하기 좋은 위치로 재배치(Scale & Shift)**해서 뒷사람에게 넘겨준다."

---

## 1. 🏜️ 직관: "모래 뿌리기" (feat. 혁펜하임)

### 💀 문제 상황: Internal Covariate Shift
딥러닝 학습은 이어달리기와 같습니다. 앞단 레이어(Layer)의 가중치($W$)가 계속 변하다 보니, 뒷단으로 넘어오는 데이터(Feature)의 분포가 계속 제멋대로 바뀝니다.
* **비유:** 들것에 실린 모래가 어떨 때는 **왼쪽 구석(음수)**에 쏠리고, 어떨 때는 **사방팔방 퍼져(분산 큼)** 들어옵니다.
* **결과:** 뒷단 레이어는 "아니, 이번엔 또 어디로 튈지 모르겠네?" 하며 당황해서 학습 속도가 느려집니다.

### 🛡️ 해결책: 2단계 재배치
BN은 데이터를 받아서 두 단계로 처리합니다.

1.  **강제 정렬 (Normalization):** 일단 모래를 **중앙(0)**으로 모으고, **적당한 폭(1)**으로 다집니다.
2.  **재배치 (Scale & Shift):** AI에게 권한을 줍니다. "네가 학습하기 제일 좋은 위치와 넓이로 **다시 뿌려봐!**"

---

## 2. 📐 핵심 원리와 수식

BN의 마법은 단순히 0으로 만드는 것(Step 1)이 아니라, **다시 흩트리는 것(Step 2)**에 있습니다.

### Step 1. Normalization (정규화)
들어온 배치 데이터($x$)의 평균과 분산을 구해 표준화합니다.
$$ \hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} $$
* 결과: 평균 0, 분산 1

### Step 2. Scale & Shift (재배치) 🔥 핵심
정규화된 데이터($\hat{x}$)에 학습 가능한 파라미터 $\gamma, \beta$를 적용합니다.
$$ y = \gamma \hat{x} + \beta $$

* **$\gamma$ (Gamma):** Scale (폭 조절) $\rightarrow$ "얼마나 세게 뿌릴까?" (분산 조절)
* **$\beta$ (Beta):** Shift (이동) $\rightarrow$ "어디 좌표에 뿌릴까?" (평균 조절)

---

## 3. 🧠 개념 확인 퀴즈 (Quizzes)

학습한 내용을 점검해 봅시다.

### Q1. 선형성의 함정
> **문제:** 왜 데이터를 정규화(Step 1)만 하고 끝내지 않고, 굳이 $\gamma, \beta$를 써서 다시 값을 망가뜨릴까요?
>
> **정답 및 해설:**
> 데이터를 무조건 0 근처(평균 0, 분산 1)에만 모아두면, **Sigmoid나 ReLU 같은 활성화 함수의 '선형(Linear) 구간'에만 데이터가 갇히게 됩니다.** (Sigmoid의 0 근처는 직선에 가깝습니다.)
> 딥러닝의 핵심은 비선형(구불구불함)을 통해 복잡한 문제를 푸는 것인데, 데이터가 선형 구간에만 있으면 층을 아무리 깊게 쌓아도 단순한 선형 모델이 되어버립니다. 그래서 $\gamma, \beta$를 통해 **"필요하다면 비선형 구간으로 데이터를 밀어버릴 수 있는(Shift)" 융통성**을 주는 것입니다.

### Q2. 파라미터 구분하기
> **문제:** 다음 중 AI가 역전파(Backprop)를 통해 **학습하는 파라미터**는 무엇인가요?
> 1. 배치의 평균($\mu$)과 분산($\sigma^2$)
> 2. 스케일($\gamma$)과 시프트($\beta$)
>
> **정답 및 해설:** **2번 ($\gamma, \beta$)**
> * $\mu, \sigma$: 그냥 들어온 데이터를 보고 계산기 두드려 구한 **통계값**입니다.
> * $\gamma, \beta$: "이 위치가 좋겠어!"라고 AI가 시행착오를 겪며 찾아내는 **학습 변수(Weight/Bias)**입니다.

### Q3. Train vs Test 시나리오
> **문제:** 테스트(Inference) 단계에서 데이터가 1개만 들어왔습니다. 이때 학습 때처럼 그 데이터 1개의 평균과 분산을 구해서 정규화하면 어떻게 될까요?
>
> **정답 및 해설:** **대참사(망함)**가 일어납니다.
> 데이터 1개의 평균은 자기 자신이고 분산은 0입니다. 분모가 0이 되어 에러가 나거나, 값이 0으로 고정되어 모델이 아무것도 예측하지 못합니다. 따라서 테스트 때는 **학습 중에 미리 적어둔 '전체 이동 평균(Running Mean/Var)'**을 꺼내 써야 합니다.

---

## 4. ⚔️ 면접 필살기: 학습 vs 추론

| 구분 | 학습 (Model.train()) | 추론/평가 (Model.eval()) |
| :--- | :--- | :--- |
| **평균/분산 기준** | **현재 들어온 배치(Batch)**의 통계값 사용 | 학습 중 누적된 **이동 평균(Running Stats)** 사용 |
| **특이사항** | 동시에 `Running Mean/Var`를 몰래 업데이트함 (컨닝 페이퍼 작성) | 저장된 `Running Mean/Var`를 꺼내서 정규화함 (컨닝 페이퍼 사용) |
| **주의점** | 배치 사이즈가 너무 작으면(예: 2, 4) 통계가 불안정함 | `state_dict` 저장 시 Running Stats도 반드시 저장해야 함 |

---

## 5. 💻 PyTorch Code 실무 적용

### 기본 코드 분석
```python
import torch.nn as nn

# 채널이 3개인 BN 레이어
bn = nn.BatchNorm1d(3)

# 1. 학습 가능한 파라미터 (Gradient 계산 O)
print(bn.weight) # Gamma (초기값 1) -> Scale
print(bn.bias)   # Beta  (초기값 0) -> Shift

# 2. 학습되지 않는 버퍼 (Gradient 계산 X) -> 테스트 때 사용
print(bn.running_mean) # 이동 평균 (초기값 0)
print(bn.running_var)  # 이동 분산 (초기값 1)
```

### 꿀팁: ConvBlock 구현 시
`Conv2d` 바로 뒤에 `BatchNorm`을 쓸 때는 **Conv의 Bias를 끕니다.**

```python
# 추천하는 구조
layer = nn.Sequential(
    # bias=False 중요! (BN의 Beta가 Bias 역할을 대신 해주므로 중복 제거)
    nn.Conv2d(64, 128, kernel_size=3, bias=False),
    
    nn.BatchNorm2d(128),  # 여기서 중심 이동(Beta)을 담당함
    
    nn.ReLU(inplace=True)
)
```

In [4]:
layer =  nn.BatchNorm1d(3)
print(layer.weight) # 표준편차 역할
print(layer.bias) # 평균 역할
print("="*20)

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=0))
print(layer(x).std(dim=0, unbiased=False)) # torch.std는 N-1로 나눔

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[-0.3488,  0.9164,  0.1914],
        [ 0.6472, -0.6477, -0.7130],
        [ 0.2866, -1.0689, -1.5645],
        [ 0.8283,  0.1077,  0.6902],
        [ 1.8950,  1.5160,  0.3816]])
tensor([[-1.3723,  0.7857,  0.4775],
        [-0.0196, -0.8492, -0.6179],
        [-0.5094, -1.2895, -1.6491],
        [ 0.2263, -0.0596,  1.0815],
        [ 1.6750,  1.4125,  0.7079]], grad_fn=<NativeBatchNormBackward0>)
tensor([ 2.0862e-08,  2.6822e-08, -2.3842e-08], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


# ⚔️ Batch Norm(BN) vs Layer Norm(LN) 완벽 비교

두 코드는 **"정규화를 하는 방향(축)"**이 서로 정반대입니다.
`dim` 파라미터가 왜 다른지 직관적으로 정리했습니다.

---

## 1. 🏫 학교 성적표 비유 (직관)

데이터 `(5, 3)`을 **학생 5명의 3과목(국어, 수학, 영어) 성적표**라고 가정합시다.

### ① BatchNorm (`dim=0`): "전교 석차 따지기"
* **관점:** "철수가 수학을 80점 받았네. **다른 애들(Batch)에 비해** 잘 본 건가?"
* **방향:** **세로 방향 ($\downarrow$)** (과목별로 학생들을 줄 세움)
* **동작:** `dim=0` (Batch 축)을 압축하여 없앰.
* **결과:** 과목 수만큼의 평균이 나옴 (국어 평균, 수학 평균, 영어 평균).

### ② LayerNorm (`dim=1`): "내 적성 찾기 (자아 성찰)"
* **관점:** "철수가 수학 80점이네. **철수의 다른 과목(Feature) 점수에 비해** 잘 본 건가?"
* **방향:** **가로 방향 ($\rightarrow$)** (학생 혼자서 자기 과목끼리 비교)
* **동작:** `dim=1` (Feature 축)을 압축하여 없앰.
* **결과:** 학생 수만큼의 평균이 나옴 (철수 평균, 영희 평균...).

---

## 2. 💻 코드 분석: 왜 dim이 다를까?

### Case 1: BatchNorm (dim=0)
```python
# BN은 같은 과목(Channel)끼리 묶어서 계산합니다.
print(layer(x).mean(dim=0))
```
* **의미:** "학생 5명의 점수를 퉁쳐서 과목별 평균을 내라."
* **결과 Shape:** `[3]` (과목이 3개니까)

### Case 2: LayerNorm (dim=1)
```python
# LN은 한 학생(Sample) 안에서 모든 과목을 묶어서 계산합니다.
print(layer(x).mean(dim=1))
```
* **의미:** "과목 3개의 점수를 퉁쳐서 학생별 평균을 내라."
* **결과 Shape:** `[5]` (학생이 5명이니까)

---

## 3. 🔍 한 눈에 보는 요약표

| 구분 | **Batch Norm (BN)** | **Layer Norm (LN)** |
| :--- | :--- | :--- |
| **비유** | **절대 평가 (남과 비교)** | **자아 성찰 (나와 비교)** |
| **방향** | **세로 ($\downarrow$)** | **가로 ($\rightarrow$)** |
| **사라지는 차원** | `dim=0` (Batch) | `dim=1` (Feature) |
| **주 사용처** | 이미지 (CNN) | **자연어 (Transformer, LLM)** |

> **💡 왜 LLM은 LayerNorm을 쓸까?**
> 문장마다 길이가 다르고 배치 사이즈가 작아도, LN은 **"나 혼자(문장 1개)"** 평균을 내기 때문에 통계가 안정적입니다. 그래서 Transformer 계열은 무조건 LN을 씁니다.

In [5]:
layer = nn.LayerNorm(3)
print(layer.weight) # 표준편차 역할
print(layer.bias) # 평균 역할

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=1))
print(layer(x).std(dim=1, unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[ 0.7581,  1.2230, -0.2036],
        [ 1.2666, -0.7588, -1.2029],
        [ 0.4689,  0.4478, -0.7050],
        [-0.0053,  1.0109,  0.3451],
        [ 0.7454,  1.9698, -0.4212]])
tensor([[ 0.2787,  1.0614, -1.3401],
        [ 1.3939, -0.4904, -0.9036],
        [ 0.7262,  0.6878, -1.4140],
        [-1.0808,  1.3302, -0.2494],
        [-0.0197,  1.2345, -1.2147]], grad_fn=<NativeLayerNormBackward0>)
tensor([ 7.9473e-08, -7.9473e-08,  0.0000e+00,  9.9341e-09,  7.9473e-08],
       grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


In [6]:
layer = nn.BatchNorm2d(3)
print(layer.weight)
print(layer.bias)

x = torch.randn(5,3,32,32)
print(layer(x).mean(dim=(0,2,3)))
print(layer(x).std(dim=(0,2,3),unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([9.3132e-11, 1.8626e-09, 2.2352e-09], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


# 📉 Dropout A to Z:

> **핵심 요약:** > "학습(Train) 때는 **강제로 연차(Dropout)**를 보내서 자생력을 기르고, 실전(Test) 때는 **전원 출근**시켜서 풀 파워를 낸다."

---

## 1. 🏢 직관: "회사가 잘 돌아가는 비결" (The Intuition)

### 💀 문제 상황: Co-adaptation (상호 의존)
* **상황:** 회사에 일을 엄청 잘하는 '김 대리' 한 명이 있습니다.
* **부작용:** 나머지 직원들은 김 대리만 믿고 묻어가려고 합니다. (Free-riding)
* **결과:** 김 대리가 아프면 회사가 마비됩니다. 딥러닝 모델로 치면 특정 노드에 **과적합(Overfitting)**된 상태입니다.

### 🛡️ 해결책: 강제 연차 (Dropout)
* **Training (연습):** 사장님이 매일 아침 동전을 던져서 **직원의 절반($p$)을 랜덤하게 집에 보냅니다.**
    * 남은 사람들끼리 어떻게든 일을 처리해야 하므로, **'김 대리' 없이도 돌아가는 법**을 배웁니다.
    * 직원들이 각자 **1인분 이상의 능력(개성 있는 특징)**을 갖게 됩니다.
* **Test (실전):** **"대통령이 방문했습니다!"** 👔
    * 중요한 날이니 **전원 출근**시킵니다.
    * 각자 능력이 올라간 상태에서 다 같이 일하니까 퍼포먼스가 최상이 됩니다. (앙상블 효과)

---

## 2. 🎚️ 메커니즘 & 수식: "성량 조절의 비밀"

연습 때는 50명만 노래 부르다가, 실전에서 100명이 다 부르면 **소리(출력값)가 2배로 커지는 문제**가 발생합니다. 이를 해결하는 방법은 두 가지입니다.

### 1) Original Paper 방식 (직관적)
* **Train:** 그냥 50명만 부름.
* **Test:** 인원이 2배가 됐으니, **"각자 목소리를 절반($\times p$)으로 줄여!"**라고 명령.
* *단점:* 테스트할 때마다 계산해야 해서 번거로움.

### 2) PyTorch 방식 (Inverted Dropout) ⭐ **[실제 사용]**
* **Train:** "너네 인원 적으니까, 연습 때 미리 **목소리를 2배($\times \frac{1}{p}$)로 키워서 불러!**" (Scaling Up)
* **Test:** **아무것도 건드리지 않음.** (그냥 전원 출근해서 평소대로 부름)
* *장점:* 실전(Inference) 속도가 빠르고 코드가 깔끔함.

---

## 3. 🖼️ 시각화: "노드의 개성" (Autoencoder 실험)

혁펜하임님이 MNIST(손글씨) 오토인코더의 은닉층(Hidden Layer)을 시각화했을 때의 차이입니다.

| 구분 | **Dropout 미적용 (Without)** | **Dropout 적용 (With)** |
| :--- | :--- | :--- |
| **이미지** | 자글자글한 노이즈 (TV 화면 지지직) | 뚜렷한 특징 (선, 곡선, 획) |
| **상태** | **눈치 게임 중 (Co-adaptation)**<br>"옆 사람이 해주겠지" 하고 대충 학습함. | **전문가 등극**<br>"내가 없으면 안 돼!"라는 마인드로<br>각자 확실한 특징(Feature)을 담당함. |

---

## 4. 💻 PyTorch 실전 코드 & 주의사항

### 핵심 코드
```python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    # p=0.5: 50% 확률로 노드를 꺼버림 (0으로 만듦)
    # PyTorch는 이때 살아남은 값을 2배로 튀기기(Scaling)까지 자동으로 해줌!
    nn.Dropout(p=0.5),
    nn.Linear(256, 10)
)
```

### 🚨 개발자의 악몽 (가장 많이 하는 실수)

> **Q. 똑같은 사진을 넣었는데 결과가 계속 바뀌어요! 왜 이러죠?**
> **A. `model.eval()`을 안 썼기 때문입니다.**

* **`model.train()`:** Dropout 켜짐 (랜덤 연차 + 목소리 뻥튀기 ON)
* **`model.eval()`:** Dropout 꺼짐 (전원 출근 + 목소리 뻥튀기 OFF)

👉 **추론(Inference)이나 테스트를 할 때는 반드시 `model.eval()`을 선언해야 합니다.**

---

## 5. 📝 3줄 요약

1.  **목적:** 특정 뉴런 편애(과적합)를 막고, 모든 뉴런을 **정예 요원(Feature Extractor)**으로 만들기 위함.
2.  **동작:** PyTorch는 **학습 때 값을 키워놓고(Inverted)**, 테스트 때는 아무 짓도 안 한다.
3.  **주의:** 실전에서 **`model.eval()`** 빼먹으면 사장님(사용자)한테 혼난다.

In [13]:
# Dropout은 논문과 구현이 다르다
x = torch.randn(3,7)
drop = nn.Dropout(p=0.7) # 구현에서 p는 죽일 확률

print(x)
print(drop(x)) # 구현은 반대로 훈련 때 1/살릴확률을 곱하고 테스트 때는 그대로

drop.eval()
print(drop(x))

"""
nn.Linear(10,100)
nn.BatcnhNorm1d(100)
nn.ReLU()
nn.Dropout(p=0.5) 이런 식으로 구성!
"""

tensor([[ 0.7131,  0.0793,  0.9172, -1.5960, -0.3188,  0.9031,  0.6755],
        [-0.7698,  0.8509, -0.5761, -0.4349,  0.9503,  0.6055, -0.1647],
        [ 1.4328, -1.2720, -0.1861,  0.4898, -1.3679, -1.5293, -1.1711]])
tensor([[ 0.0000,  0.2644,  0.0000, -0.0000, -0.0000,  0.0000,  0.0000],
        [-0.0000,  0.0000, -1.9205, -0.0000,  0.0000,  0.0000, -0.0000],
        [ 0.0000, -4.2401, -0.6204,  0.0000, -0.0000, -0.0000, -0.0000]])
tensor([[ 0.7131,  0.0793,  0.9172, -1.5960, -0.3188,  0.9031,  0.6755],
        [-0.7698,  0.8509, -0.5761, -0.4349,  0.9503,  0.6055, -0.1647],
        [ 1.4328, -1.2720, -0.1861,  0.4898, -1.3679, -1.5293, -1.1711]])


'\nnn.Linear(10,100)\nnn.BatcnhNorm1d(100)\nnn.ReLU()\nnn.Dropout(p=0.5) 이런 식으로 구성!\n'

In [17]:
class sample_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop_layer = nn.Sequential(
            nn.Linear(5,7),
            nn.ReLU(),
            nn.Dropout(p=0.9)
        )

    def forward(self,x):
        x = self.drop_layer(x)
        return x

model = sample_model()
model.train() # train mode로 전환
x = torch.randn(2,3,5)
print(model(x))

model.eval() # test mode
print(model(x)) # 일부 0인 이유: 음수인 애들이 ReLU 때문에 사라짐.

tensor([[[ 6.0379,  0.0000,  0.0000,  0.0000,  1.1134,  4.9395,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [13.8675,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<MulBackward0>)
tensor([[[0.6038, 0.3826, 0.0000, 0.8605, 0.1113, 0.4939, 1.0091],
         [0.0000, 0.8718, 0.3004, 0.7212, 0.0382, 0.0789, 0.4232],
         [0.0000, 0.3210, 0.1651, 0.4883, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.5744, 0.1951, 0.4173, 0.2951, 0.0000, 0.0456],
         [0.0000, 0.0405, 0.0000, 0.2490, 0.8108, 0.0498, 0.2351],
         [1.3868, 0.2497, 0.0000, 0.2343, 0.0000, 0.2545, 1.1685]]],
       grad_fn=<ReluBackward0>)
